# SAINSC segmentation-free analysis of ISS data

This notebook converts decoded ISS transcripts into a physically calibrated SAINSC grid and exports local expression-density maxima as an AnnData object. These maxima are **cell proxies**, not segmented cell boundaries.

SAINSC runs in a separate Python 3.12 environment. Create it once with:

```bash
conda env create -f ISS_postprocessing/environments/sainsc.yml
```

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

import ISS_postprocessing.segmentation_free as SEGFREE

## Dataset and physical scale

`pixel_size_um` must describe the image coordinate system used by `xc` and `yc`. If the decoded coordinates are already micrometres, set `coordinate_unit = "um"`.

In [ ]:
input_dir = Path("/path/to/experiment")
region = "R1"
pixel_size_um = 0.325
coordinate_unit = "pixel"

transcripts = input_dir / region / "decoding" / "2_decoded" / f"{region}_decoded.csv"
spots = pd.read_csv(transcripts)
display(spots.head())
print(f"{len(spots):,} transcripts; {spots['target'].nunique():,} genes")

In [ ]:
# A quick density check before selecting physical-scale parameters.
sample = spots.sample(min(len(spots), 100_000), random_state=0)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(sample['xc'], sample['yc'], s=0.2, alpha=0.4)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_title(f"{region}: decoded transcript density")
ax.set_xlabel('x (source coordinates)')
ax.set_ylabel('y (source coordinates)');

## Prepare without running SAINSC

Preparation normalizes `target/xc/yc`, applies optional quality filtering, and writes a complete runner configuration. It does not require SAINSC or a GPU. Start without a background threshold, inspect the first result, and then tune it if low-density noise creates excessive maxima.

In [ ]:
parameters = dict(
    grid_size_um=0.5,
    kernel_bandwidth_um=4.0,
    min_peak_distance_um=4.0,
    min_peak_area_um2=0.0,
    background_min_norm=None,
    quality_col=None,          # e.g. 'quality_minimum'
    min_quality=None,          # e.g. 0.5
    n_threads=8,
    save_maps=False,           # dense maps can be large
)

prepared = SEGFREE.prepare_sainsc_analysis(
    transcripts,
    input_dir=input_dir,
    region=region,
    coordinate_unit=coordinate_unit,
    pixel_size_um=pixel_size_um,
    **parameters,
)
print(json.dumps(json.loads(prepared['config'].read_text()), indent=2))

## Run SAINSC

For reference-based cell typing, set `signatures` to a CSV or DataFrame whose rows are genes and columns are cell types. Leave it as `None` for local-maxima discovery alone.

In [ ]:
signatures = None  # Path('/path/to/gene_by_celltype_signatures.csv')

outputs = SEGFREE.run_sainsc_analysis(
    transcripts,
    input_dir=input_dir,
    region=region,
    coordinate_unit=coordinate_unit,
    pixel_size_um=pixel_size_um,
    signatures=signatures,
    sainsc_python=["conda", "run", "-n", "sainsc", "python"],
    overwrite=False,
    **parameters,
)
outputs

## Inspect cell proxies

The standardized `spatial` coordinates are image pixels; `spatial_um` contains micrometres and `sainsc_grid` preserves native grid indices. Review proxy density and expression before treating maxima as biological cells.

In [ ]:
adata = sc.read_h5ad(outputs['local_maxima'])
print(adata)
print(adata.uns['sainsc'])

coords = adata.obsm['spatial']
fig, ax = plt.subplots(figsize=(7, 7))
if 'celltype' in adata.obs:
    categories = adata.obs['celltype'].astype('category')
    scatter = ax.scatter(coords[:, 0], coords[:, 1], c=categories.cat.codes, s=2, cmap='tab20')
    handles, _ = scatter.legend_elements()
    ax.legend(handles, categories.cat.categories, title='cell type', bbox_to_anchor=(1.02, 1), loc='upper left')
else:
    values = adata.obs.get('total_mRNA_KDE')
    scatter = ax.scatter(coords[:, 0], coords[:, 1], c=values, s=2, cmap='magma')
    fig.colorbar(scatter, ax=ax, label='total mRNA KDE')
ax.invert_yaxis()
ax.set_aspect('equal')
ax.set_title(f"{region}: SAINSC local-maxima proxies");